In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F

# Read from Bronze
df = spark.read.parquet("abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Bronze.Lakehouse/Files/yellow_tripdata_2024-01.parquet")

# Clean: remove invalid trips
df_silver = df.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") > 0) &
    (F.col("passenger_count") > 0) &
    (F.col("tpep_pickup_datetime").isNotNull())
).withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
 .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
 .withColumn("trip_duration_min",
    F.round((F.unix_timestamp("tpep_dropoff_datetime") -
             F.unix_timestamp("tpep_pickup_datetime")) / 60, 2))

print(f"Original: {df.count()} rows")
print(f"Silver:   {df_silver.count()} rows")
print(f"Dropped:  {df.count() - df_silver.count()} bad rows")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [2]:
# Write clean taxi data to Silver as Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_taxi")

print("silver_taxi table written successfully")

StatementMeta(, 111da7b5-dc72-4358-a548-85f93f9dbfa2, 4, Finished, Available, Finished, False)

silver_taxi table written successfully


In [3]:
from pyspark.sql import functions as F

BRONZE = "abfss://FabricDataProject@onelake.dfs.fabric.microsoft.com/Bronze.Lakehouse/Files"

# ── 1. OPENAQ PM2.5 ──────────────────────────────────────
df_aq_raw = spark.read.option("multiline", "true").json(f"{BRONZE}/openaq_nyc_pm25.json")

df_aq = df_aq_raw.select(
    F.col("period.datetimeFrom.utc").alias("datetime"),
    F.col("value").cast("double").alias("pm25_value")
).withColumn("datetime", F.to_timestamp("datetime")) \
 .withColumn("date", F.to_date("datetime")) \
 .withColumn("hour", F.hour("datetime")) \
 .filter(F.col("pm25_value") >= 0)

df_aq.write.format("delta").mode("overwrite").saveAsTable("silver_air_quality")
print(f"silver_air_quality: {df_aq.count()} rows")

# ── 2. WORLD BANK GDP ────────────────────────────────────
df_gdp_raw = spark.read.option("multiline", "true").json(f"{BRONZE}/usa_gdp.json")
df_gdp = df_gdp_raw.select(
    F.col("date").cast("integer").alias("year"),
    F.col("value").cast("double").alias("gdp_usd")
).filter(F.col("gdp_usd").isNotNull()).orderBy("year")

df_gdp.write.format("delta").mode("overwrite").saveAsTable("silver_gdp")
print(f"silver_gdp: {df_gdp.count()} rows")

# ── 3. ECB FX RATES ──────────────────────────────────────
df_fx = spark.read.csv(f"{BRONZE}/ecb_fx_usd_eur.csv", header=True).select(
    F.to_date("TIME_PERIOD").alias("date"),
    F.col("OBS_VALUE").cast("double").alias("usd_per_eur")
).filter(F.col("usd_per_eur").isNotNull())

df_fx.write.format("delta").mode("overwrite").saveAsTable("silver_fx")
print(f"silver_fx: {df_fx.count()} rows")

StatementMeta(, 111da7b5-dc72-4358-a548-85f93f9dbfa2, 5, Finished, Available, Finished, False)

silver_air_quality: 1000 rows
silver_gdp: 10 rows
silver_fx: 6992 rows
